# Intro 1：TabularX 快速入门 —— 二分类任务

本教程带你快速上手 TabularX 的核心功能：**一键建模、自动报告、可解释性与公平性评估**。

### 你将学到
- 如何用  一键完成完整建模流水线
- 如何配置各阶段参数
- 如何查看和保存实验结果
- 如何检查模型的公平性

### 数据集
使用 sklearn 内置的 **Breast Cancer 数据集**（二分类：恶性/良性）。

In [ ]:
# 导入
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from sklearn.datasets import load_breast_cancer

import tabularx as tbx

print("TabularX 已就绪")

In [ ]:
# 加载数据
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target

print(f"特征维度: {X.shape}")
print(f"样本数: {len(X)}")
print(f"特征数: {len(X.columns)}")
print("目标分布:")
print(y.value_counts())
print(f"特征列表: {list(X.columns[:5])} ...（共 {len(X.columns)} 个特征）")


## 第一步：创建 AutoEstimator

 是 TabularX 的高阶入口。你只需指定任务类型和输出目录，框架会自动完成：
1. **Profile** — 数据概况分析
2. **Split** — 数据划分（train/valid/test）
3. **Preprocess** — 缺失值填补、编码、标准化
4. **Model Search** — 多模型并行搜索与调优
5. **Voting** — 最优模型组合
6. **SHAP 解释** — 特征重要性分析
7. **公平性评估** — 敏感子组性能差异分析（可选）

我们还可以通过  对象灵活配置各阶段参数。

In [ ]:
# 创建估计器
ae = tbx.AutoEstimator(task="classification", output_dir="./intro1-output")

# 配置参数
ae.param.model_search.set_param("n_trials", 30)
ae.param.model_search.set_param("top_k_models", 2)
ae.param.shap.set_param("top_k_features", 10)
ae.param.split.set_param("test_size", 0.2)
ae.param.split.set_param("validation_size", 0.25)

print("参数总览:\n")
print(ae.param.overview())

## 第二步：一键运行完整流水线

调用 `fit()` 方法自动执行所有阶段。运行时间取决于 `n_trials` 和机器配置，大约 1-2 分钟。

In [ ]:
# 为公平性评估创建分组特征
X["radius_group"] = pd.cut(X["mean radius"], bins=3,
                             labels=["small", "medium", "large"])

# 一键运行
ae.fit(X, y, sensitive_features=["radius_group"])

print("运行完成！")
print(f"输出目录: {ae.output_dir}")

## 第三步：查看搜索结果

`fit()` 运行后，所有结果存储在 `ae.result`（`AutoResult` 对象）中。

In [ ]:
result = ae.result

print(f"任务类型: {result.searched.task_type}")
print(f"主指标: {result.searched.primary_metric} （{('越大越好' if result.searched.optimization_direction == 'maximize' else '越小越好')}）\n")

print("=" * 70)
print(f"{'排名':>4} {'模型':>20} {'验证集':>10} {'测试集':>10}")
print("=" * 70)
for i, r in enumerate(result.searched.ranking, 1):
    valid = r.validation_metrics.get(result.searched.primary_metric, float("nan"))
    test = (r.test_metrics or {}).get(result.searched.primary_metric, float("nan"))
    print(f"{i:>4} {r.model_name:>20} {valid:>10.4f} {test:>10.4f}")

print(f"\n最佳模型: {result.searched.best_model_name}")

## 第四步：查看 SHAP 可解释性

SHAP 分析自动对最优模型执行，计算每个特征对预测的贡献。

In [ ]:
shap = result.shap

if shap is not None:
    print(f"模型: {shap.model_name}")
    print(f"解释样本数: {shap.n_explain_samples}")
    print(f"Top-{len(shap.top_k_features)} 特征解释力占比: {shap.top_k_explained_ratio:.1%}\n")

    print(f"{'排名':>4} {'特征名':>25} {'平均 |SHAP|':>12}")
    print("-" * 45)
    for i, feat in enumerate(shap.top_k_features, 1):
        val = shap.feature_importance.get(feat, 0.0)
        print(f"{i:>4} {feat:>25} {val:>12.4f}")
else:
    print("SHAP 未执行")

## 第五步：公平性评估

TabularX 自动检查模型在不同敏感特征子组上的性能差异。

注意：Breast Cancer 数据集的特征都是连续数值，不适合直接做公平性分析。
这里我们先将 `mean radius` 分为 small / medium / large 三组，
模拟一个类别型敏感特征来演示公平性审计能力。

In [ ]:
# 公平性评估结果
fairness = result.fairness

if fairness is not None and fairness.enabled:
    for feat_result in fairness.feature_results:
        print(f"敏感特征: {feat_result.feature_name}")
        if feat_result.is_dropped:
            print(f"  已丢弃: {feat_result.dropped_reason}")
            continue

        print(f"  DP差异: {feat_result.demographic_parity_diff:.4f}" +
              f" {'告警' if feat_result.dp_violation else '正常'}")
        print(f"  EOdds TPR差异: {feat_result.equalized_odds_tpr_diff:.4f}" +
              f" {'告警' if feat_result.eo_tpr_violation else '正常'}")
        print(f"  EOdds FPR差异: {feat_result.equalized_odds_fpr_diff:.4f}" +
              f" {'告警' if feat_result.eo_fpr_violation else '正常'}")

        print(f"\n  {'子组':>16} {'样本数':>8} {'正类率':>10} {'准确率':>10} {'TPR':>10} {'FPR':>10}")
        print("  " + "-" * 66)
        for g in feat_result.groups:
            print(f"  {g.group_name:>16} {g.n_samples:>8} {g.positive_rate:>10.4f} "
                  f"{g.accuracy:>10.4f} {g.tpr:>10.4f} {g.fpr:>10.4f}")
else:
    print("未启用公平性评估。")

## 第六步：查看生成的报告

TabularX 强制生成可复现实验报告。所有 HTML 报告已保存在输出目录中，可以直接用浏览器打开查看。

In [ ]:
# 报告文件列表
from pathlib import Path

print("生成的文件:")
for name, path in result.report_paths.items():
    exists = "OK" if Path(path).exists() else "MISS"
    print(f"  [{exists}] {name}: {path}")

## 总结

本教程演示了 TabularX 的二分类端到端流程：
1. 创建 `AutoEstimator` 并配置参数
2. 一键运行完整流水线（Profile -> Split -> Preprocess -> Search -> SHAP -> Fairness）
3. 查看模型排名和 SHAP 特征重要性
4. 检查公平性审计结果（敏感子组的 DP / EOdds 差异）
5. 自动生成 HTML 报告

TabularX 的所有阶段也可单独使用底层函数。
下一教程将展示**多分类任务**和**参数深度调优**。